# SET UP

In [1]:
# =========================
# Cell 1 - Import libraries and install dependencies
# =========================
import os
import json
import math
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from collections import Counter

# Try to import transformers; install if missing
try:
    from transformers import CLIPProcessor, CLIPModel
except Exception as e:
    print("Installing transformers...")
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "transformers"])
    from transformers import CLIPProcessor, CLIPModel

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [52]:
# =========================
# Cell 2 - Directory variables (set paths here)
# =========================
# UPDATE THIS PATH to point to where you imported the BERTopic notebook's output
DATA_CSV = "/kaggle/input/datasets/namanne/flickr30k-bertopic-labels/flickr30k_labeled_confident.csv"  
# Standard Flickr30k Kaggle path
IMAGE_ROOT = "/kaggle/input/datasets/hsankesara/flickr-image-dataset/"      
OUTPUT_DIR = "/kaggle/working"

LABEL_MAPPING_JSON = None  

MAX_SAMPLES = None          
VAL_SIZE = 0.1
TEST_SIZE = 0.1

FEW_SHOT_K = 100
BATCH_SIZE = 64
EPOCHS = 15
LR = 1e-3

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("OUTPUT_DIR:", OUTPUT_DIR)

OUTPUT_DIR: /kaggle/working


In [3]:
# =========================
# Cell 3 - Load labeled dataset
# =========================
df = pd.read_csv(DATA_CSV)

# ADAPTATION: Rename the column from the BERTopic pipeline to match what CLIP expects
if "caption_first" in df.columns:
    df = df.rename(columns={"caption_first": "caption"})

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head(3))

required_cols = ["image_path", "caption", "label"]
for c in required_cols:
    if c not in df.columns:
        raise ValueError(f"Missing required column: {c}")

if MAX_SAMPLES:
    df = df.sample(MAX_SAMPLES, random_state=42).reset_index(drop=True)

# Create label_id if missing
if "label_id" not in df.columns:
    labels_sorted = sorted(df["label"].unique())
    label_to_id = {l: i for i, l in enumerate(labels_sorted)}
    df["label_id"] = df["label"].map(label_to_id)
else:
    labels_sorted = sorted(df["label"].unique())
    label_to_id = {l: i for i, l in enumerate(labels_sorted)}

print("Num classes:", len(label_to_id))
print(label_to_id)

Shape: (30845, 63)
Columns: ['image_id', 'image_path', 'image_file', 'label_final', 'label_coarse', 'zs_label', 'zs_score', 'label_uncertain', 'bertopic_id', 'caption', 'captions', 'label', 'label_id', 'zs_score_outdoor_adventure', 'zs_score_outdoor_adventure.1', 'zs_score_photography', 'zs_score_children', 'zs_score_children.1', 'zs_score_sports', 'zs_score_sports.1', 'zs_score_music_performance', 'zs_score_music_performance.1', 'zs_score_water_boats', 'zs_score_water_boats.1', 'zs_score_food_and_market', 'zs_score_food_and_market.1', 'zs_score_street_walking', 'zs_score_street_walking.1', 'zs_score_beach', 'zs_score_beach.1', 'zs_score_dogs', 'zs_score_dogs.1', 'zs_score_construction_work', 'zs_score_construction_work.1', 'zs_score_biking', 'zs_score_biking.1', 'zs_score_wedding', 'zs_score_wedding.1', 'zs_score_outdoor_adventure.2', 'zs_score_outdoor_adventure.3', 'zs_score_taking_pictures', 'zs_score_children.2', 'zs_score_children.3', 'zs_score_sports.2', 'zs_score_sports.3', 'zs_

,image_id,image_path,image_file,label_final,label_coarse,zs_label,zs_score,label_uncertain,bertopic_id,caption,...,zs_score_beach.2,zs_score_beach.3,zs_score_dogs.2,zs_score_dogs.3,zs_score_construction_work.2,zs_score_construction_work.3,zs_score_biking.2,zs_score_biking.3,zs_score_wedding.2,zs_score_wedding.3
0,1000092795,/kaggle/input/datasets/hsankesara/flickr-image...,1000092795.jpg,construction_work,construction_work,outdoor_adventure,0.672723,False,2,Two young guys with shaggy hair look at their ...,...,0.007605,0.011191,0.007284,0.010719,0.006772,0.009965,0.006345,0.009337,0.002418,0.003558
1,10002456,/kaggle/input/datasets/hsankesara/flickr-image...,10002456.jpg,construction_work,construction_work,construction_work,0.719118,False,2,Several men in hard hats are operating a giant...,...,0.006914,0.007083,0.002366,0.002424,0.702015,0.719118,0.002675,0.002740,0.001823,0.001867
2,1000268201,/kaggle/input/datasets/hsankesara/flickr-image...,1000268201.jpg,children,other,children,0.981141,False,-1,A child in a pink dress is climbing up a set o...,...,0.000387,0.000399,0.000325,0.000335,0.002777,0.002861,0.000434,0.000447,0.000198,0.000204


Num classes: 13
{'beach': 0, 'biking': 1, 'children': 2, 'construction_work': 3, 'dogs': 4, 'food_and_market': 5, 'music_performance': 6, 'outdoor_adventure': 7, 'sports': 8, 'street_walking': 9, 'taking_pictures': 10, 'water_boats': 11, 'wedding': 12}


In [4]:
# =========================
# Cell 4 - Train/Val/Test split
# =========================
if "split" in df.columns:
    train_df = df[df["split"] == "train"].reset_index(drop=True)
    val_df = df[df["split"] == "val"].reset_index(drop=True)
    test_df = df[df["split"] == "test"].reset_index(drop=True)
else:
    train_df, temp_df = train_test_split(
        df, test_size=VAL_SIZE + TEST_SIZE, stratify=df["label"], random_state=42
    )
    val_df, test_df = train_test_split(
        temp_df, test_size=TEST_SIZE / (VAL_SIZE + TEST_SIZE), stratify=temp_df["label"], random_state=42
    )

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

Train: (24676, 63)
Val: (3084, 63)
Test: (3085, 63)


In [5]:
# =========================
# Cell 5 - Load CLIP model and processor
# =========================
MODEL_NAME = "openai/clip-vit-base-patch32"
clip_model = CLIPModel.from_pretrained(MODEL_NAME).to(device)
clip_processor = CLIPProcessor.from_pretrained(MODEL_NAME)
clip_model.eval()

print("Loaded CLIP:", MODEL_NAME)

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

Loaded CLIP: openai/clip-vit-base-patch32


In [6]:
# =========================
# Cell 6 - Dataset and DataLoader
# =========================
def safe_open_image(path):
    try:
        return Image.open(path).convert("RGB")
    except Exception:
        return Image.new("RGB", (224, 224), color=(255, 255, 255))

class FlickrCLIPDataset(Dataset):
    def __init__(self, df, image_root):
        self.df = df.reset_index(drop=True)
        self.image_root = image_root

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = row["image_path"]
        if not os.path.isabs(image_path):
            image_path = os.path.join(self.image_root, os.path.basename(image_path))
        image = safe_open_image(image_path)
        text = str(row["caption"])
        label = int(row["label_id"])
        return image, text, label

def clip_collate(batch):
    images, texts, labels = zip(*batch)
    return list(images), list(texts), torch.tensor(labels, dtype=torch.long)

def build_loader(df, shuffle=False):
    ds = FlickrCLIPDataset(df, IMAGE_ROOT)
    return DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=2,
        collate_fn=clip_collate
    )

train_loader = build_loader(train_df, shuffle=True)
val_loader = build_loader(val_df, shuffle=False)
test_loader = build_loader(test_df, shuffle=False)

print("Loaders ready.")

Loaders ready.


In [7]:
# =========================
# Cell 7 - Embedding extraction functions 
# =========================
def _to_device(batch):
    return {k: v.to(device) for k, v in batch.items()}

def clip_text_features(texts):
    inputs = clip_processor(text=list(texts), return_tensors="pt", padding=True, truncation=True)
    inputs = _to_device(inputs)
    outputs = clip_model.get_text_features(**inputs)

    if isinstance(outputs, torch.Tensor):
        feats = outputs
    elif hasattr(outputs, "text_embeds") and outputs.text_embeds is not None:
        feats = outputs.text_embeds
    elif hasattr(outputs, "pooler_output") and outputs.pooler_output is not None:
        feats = outputs.pooler_output
    else:
        feats = outputs[0]

    # Project if needed
    if hasattr(clip_model, "text_projection"):
        out_dim = clip_model.text_projection.out_features
        if feats.shape[-1] != out_dim:
            feats = clip_model.text_projection(feats)

    return F.normalize(feats, dim=-1)

def clip_image_features(images):
    inputs = clip_processor(images=list(images), return_tensors="pt")
    inputs = _to_device(inputs)
    outputs = clip_model.get_image_features(**inputs)

    if isinstance(outputs, torch.Tensor):
        feats = outputs
    elif hasattr(outputs, "image_embeds") and outputs.image_embeds is not None:
        feats = outputs.image_embeds
    elif hasattr(outputs, "pooler_output") and outputs.pooler_output is not None:
        feats = outputs.pooler_output
    else:
        feats = outputs[0]

    # Project if needed
    if hasattr(clip_model, "visual_projection"):
        out_dim = clip_model.visual_projection.out_features
        if feats.shape[-1] != out_dim:
            feats = clip_model.visual_projection(feats)

    return F.normalize(feats, dim=-1)

@torch.no_grad()
def extract_embeddings(loader, mode="image"):
    all_feats = []
    all_labels = []

    for images, texts, labels in loader:
        if mode in ["image", "multimodal"]:
            img_feats = clip_image_features(images)
        if mode in ["text", "multimodal"]:
            txt_feats = clip_text_features(texts)

        if mode == "image":
            feats = img_feats
        elif mode == "text":
            feats = txt_feats
        else:
            feats = torch.cat([img_feats, txt_feats], dim=-1)

        all_feats.append(feats.cpu())
        all_labels.append(labels)

    X = torch.cat(all_feats, dim=0)
    y = torch.cat(all_labels, dim=0)
    return X, y

# ZEROSHOT

In [8]:
# =========================
# Cell 8 - Zero-shot classification
# =========================
label_names = sorted(label_to_id.keys())

PROMPT_TEMPLATES = [
    "a photo of {label}",
    "a photo of a {label}",
    "a photo of the {label}",
    "a close-up photo of {label}",
    "a picture of {label}",
    "a photo of {label} in the scene"
]

label_prompts = [
    [tmpl.format(label=lbl.replace("_", " ")) for tmpl in PROMPT_TEMPLATES]
    for lbl in label_names
]

@torch.no_grad()
def get_label_text_embeddings():
    label_embeds = []
    for prompts in label_prompts:
        embeds = clip_text_features(prompts)
        embeds = F.normalize(embeds, dim=-1)
        mean_embed = embeds.mean(dim=0, keepdim=True)
        mean_embed = F.normalize(mean_embed, dim=-1)
        label_embeds.append(mean_embed)
    return torch.cat(label_embeds, dim=0)

label_text_embeds = get_label_text_embeddings()

@torch.no_grad()
def zero_shot_predict(loader, mode="image"):
    preds = []
    labels = []

    for images, texts, y in loader:
        if mode == "image":
            img_feats = clip_image_features(images)
            sims = img_feats @ label_text_embeds.T

        elif mode == "text":
            txt_feats = clip_text_features(texts)
            sims = txt_feats @ label_text_embeds.T

        else:
            img_feats = clip_image_features(images)
            txt_feats = clip_text_features(texts)
            fused = F.normalize((img_feats + txt_feats) / 2.0, dim=-1)
            sims = fused @ label_text_embeds.T

        pred = sims.argmax(dim=1).cpu().numpy()
        preds.extend(pred.tolist())
        labels.extend(y.numpy().tolist())

    return np.array(labels), np.array(preds)

In [9]:
# =========================
# Cell 9 - Zero-shot evaluation
# =========================
def eval_metrics(y_true, y_pred, name=""):
    acc = accuracy_score(y_true, y_pred)
    f1m = f1_score(y_true, y_pred, average="macro")
    f1w = f1_score(y_true, y_pred, average="weighted")
    print(f"{name} Acc={acc:.4f} F1_macro={f1m:.4f} F1_weighted={f1w:.4f}")
    return acc, f1m, f1w

print("Zero-shot Image")
y_true, y_pred = zero_shot_predict(test_loader, mode="image")
zs_img = eval_metrics(y_true, y_pred, "Zero-shot Image")

print("Zero-shot Text")
y_true, y_pred = zero_shot_predict(test_loader, mode="text")
zs_txt = eval_metrics(y_true, y_pred, "Zero-shot Text")

print("Zero-shot Multimodal")
y_true, y_pred = zero_shot_predict(test_loader, mode="multimodal")
zs_mm = eval_metrics(y_true, y_pred, "Zero-shot Multimodal")

Zero-shot Image
Zero-shot Image Acc=0.5378 F1_macro=0.5117 F1_weighted=0.5368
Zero-shot Text
Zero-shot Text Acc=0.5125 F1_macro=0.5009 F1_weighted=0.4940
Zero-shot Multimodal
Zero-shot Multimodal Acc=0.5874 F1_macro=0.5696 F1_weighted=0.5776


# FEWSHOT (MLP)

In [53]:
# =========================
# Cell 10 - Few-shot subset sampling 
# =========================
def few_shot_subset(df, k=FEW_SHOT_K):
    few = []
    for lbl, group in df.groupby("label"):
        few.append(group.sample(min(k, len(group)), random_state=42))
    return pd.concat(few).reset_index(drop=True)

few_train_df = few_shot_subset(train_df, FEW_SHOT_K)

# CRITICAL FIX: shuffle=False ensures the labels align perfectly 
# when we extract Image, Text, and Multimodal embeddings sequentially!
few_train_loader = build_loader(few_train_df, shuffle=False)

print("Few-shot train size:", few_train_df.shape)

Few-shot train size: (1300, 63)


In [54]:
# =========================
# Cell 11 - Extract embeddings for few-shot training
# =========================
# Few-shot train embeddings
X_train_img, y_train = extract_embeddings(few_train_loader, mode="image")
X_train_txt, _ = extract_embeddings(few_train_loader, mode="text")
X_train_mm, _ = extract_embeddings(few_train_loader, mode="multimodal")

# Val embeddings
X_val_img, y_val = extract_embeddings(val_loader, mode="image")
X_val_txt, _ = extract_embeddings(val_loader, mode="text")
X_val_mm, _ = extract_embeddings(val_loader, mode="multimodal")

# Test embeddings
X_test_img, y_test = extract_embeddings(test_loader, mode="image")
X_test_txt, _ = extract_embeddings(test_loader, mode="text")
X_test_mm, _ = extract_embeddings(test_loader, mode="multimodal")

print("Embeddings extracted.")

Embeddings extracted.


In [55]:
# =========================
# Cell 12 - MLP classifier training (few-shot) 
# =========================
class MLPClassifier(nn.Module):
    def __init__(self, input_dim, num_classes, hidden_dim=128):  # Moderate capacity
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),         # Stabilizes text embeddings
            nn.ReLU(),
            nn.Dropout(0.4),                  # High dropout
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x):
        return self.net(x)

class EmbeddingDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

def train_mlp_classifier(X_train, y_train, X_val, y_val, epochs=EPOCHS):
    model = MLPClassifier(X_train.shape[1], len(label_to_id)).to(device)
    
    # We keep light L2 weight decay for additional safety
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
    
    # CRITICAL FIX: We apply strong label smoothing. 
    # This prevents the model from over-trusting individual training examples.
    criterion = nn.CrossEntropyLoss(label_smoothing=0.2)

    train_ds = EmbeddingDataset(X_train, y_train)
    train_loader = DataLoader(train_ds, batch_size=min(256, len(train_ds)), shuffle=True)

    X_val = X_val.to(device)
    y_val = y_val.to(device)

    for ep in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            preds = torch.argmax(model(X_val), dim=1).cpu().numpy()
            acc = accuracy_score(y_val.cpu().numpy(), preds)
            f1m = f1_score(y_val.cpu().numpy(), preds, average="macro")
            train_preds = torch.argmax(model(X_train.to(device)), dim=1).cpu().numpy()
            train_acc = accuracy_score(y_train.cpu().numpy(), train_preds)
        
        # Reduced printing for cleaner output
        if ep % 5 == 0 or ep == epochs - 1:
            print(f"Epoch {ep+1} loss={loss.item():.4f} train_acc={train_acc:.4f} val_acc={acc:.4f} val_f1m={f1m:.4f}")

    return model

In [56]:
# =========================
# Cell 13 - Few-shot training & evaluation (Image/Text/Multimodal)
# =========================
print("Few-shot Image")
img_clf = train_mlp_classifier(X_train_img, y_train, X_val_img, y_val)
img_preds = torch.argmax(img_clf(X_test_img.to(device)), dim=1).cpu().numpy()
fs_img = eval_metrics(y_test.numpy(), img_preds, "Few-shot Image")

print("Few-shot Text")
txt_clf = train_mlp_classifier(X_train_txt, y_train, X_val_txt, y_val)
txt_preds = torch.argmax(txt_clf(X_test_txt.to(device)), dim=1).cpu().numpy()
fs_txt = eval_metrics(y_test.numpy(), txt_preds, "Few-shot Text")

print("Few-shot Multimodal")
mm_clf = train_mlp_classifier(X_train_mm, y_train, X_val_mm, y_val)
mm_preds = torch.argmax(mm_clf(X_test_mm.to(device)), dim=1).cpu().numpy()
fs_mm = eval_metrics(y_test.numpy(), mm_preds, "Few-shot Multimodal")

Few-shot Image
Epoch 1 loss=2.3893 train_acc=0.5100 val_acc=0.4261 val_f1m=0.3954
Epoch 6 loss=1.5481 train_acc=0.7792 val_acc=0.6485 val_f1m=0.6256
Epoch 11 loss=1.4922 train_acc=0.8692 val_acc=0.6608 val_f1m=0.6430
Epoch 15 loss=1.2080 train_acc=0.9208 val_acc=0.6657 val_f1m=0.6571
Few-shot Image Acc=0.6752 F1_macro=0.6562 F1_weighted=0.6789
Few-shot Text
Epoch 1 loss=2.4061 train_acc=0.5185 val_acc=0.4066 val_f1m=0.3934
Epoch 6 loss=1.6484 train_acc=0.8169 val_acc=0.6702 val_f1m=0.6370
Epoch 11 loss=1.3178 train_acc=0.8915 val_acc=0.7124 val_f1m=0.6795
Epoch 15 loss=1.4458 train_acc=0.9285 val_acc=0.7111 val_f1m=0.6804
Few-shot Text Acc=0.7293 F1_macro=0.7031 F1_weighted=0.7286
Few-shot Multimodal
Epoch 1 loss=2.1657 train_acc=0.6808 val_acc=0.5723 val_f1m=0.5538
Epoch 6 loss=1.2508 train_acc=0.8892 val_acc=0.7075 val_f1m=0.6951
Epoch 11 loss=1.1442 train_acc=0.9592 val_acc=0.7270 val_f1m=0.7050
Epoch 15 loss=1.0707 train_acc=0.9862 val_acc=0.7237 val_f1m=0.7082
Few-shot Multimodal 

# FEWSHOT (COOP)

In [57]:
# =========================
# Cell 14 - Improved CoOp prompt learning (few-shot across modalities)
# =========================
COOP_CLASS_SPECIFIC = True
COOP_EPOCHS = EPOCHS                 
COOP_WEIGHT_DECAY = 5e-4          
COOP_LABEL_SMOOTHING = 0.1        
COOP_BATCH_SIZE = 32              

COOP_CTX_INIT_BY_MODALITY = {
    "Image": "a photo of",
    "Text": "a caption of",       
    "Multimodal": "a photo of",   
}
COOP_LR_BY_MODALITY = {
    "Image": 0.002,              
    "Text": 0.001,
    "Multimodal": 0.001,
}

for param in clip_model.parameters():
    param.requires_grad = False
clip_model.eval()

def make_coop_multimodal_features(img_features, txt_features):
    return F.normalize((img_features + txt_features) / 2.0, dim=-1)

class CoOpPromptLearner(nn.Module):
    def __init__(self, class_names, clip_model, tokenizer, ctx_init, class_specific_context=COOP_CLASS_SPECIFIC):
        super().__init__()
        self.clip_model = clip_model
        self.class_names = [name.replace("_", " ") for name in class_names]
        self.class_specific_context = class_specific_context
        self.num_classes = len(self.class_names)

        tokenized_ctx = tokenizer(ctx_init, return_tensors="pt")
        self.n_ctx = int(tokenized_ctx["attention_mask"][0].sum().item() - 2)
        if self.n_ctx <= 0:
            raise ValueError("ctx_init must tokenize to at least one context token.")

        prompt_texts = [f"{ctx_init} {name}." for name in self.class_names]
        tokenized = tokenizer(
            prompt_texts,
            padding="max_length",
            truncation=True,
            max_length=clip_model.config.text_config.max_position_embeddings,
            return_tensors="pt",
        )
        text_device = clip_model.text_model.embeddings.token_embedding.weight.device
        tokenized = {k: v.to(text_device) for k, v in tokenized.items()}
        self.register_buffer("input_ids", tokenized["input_ids"])
        self.register_buffer("attention_mask", tokenized["attention_mask"])

        with torch.no_grad():
            base_embeds = clip_model.text_model.embeddings.token_embedding(self.input_ids)
            ctx_embed = base_embeds[0, 1:1 + self.n_ctx, :].clone()

        if class_specific_context:
            ctx_embed = ctx_embed.unsqueeze(0).repeat(self.num_classes, 1, 1)
        self.ctx = nn.Parameter(ctx_embed)

    def _build_causal_mask(self, batch_size, seq_len, dtype, device):
        mask = torch.full((seq_len, seq_len), torch.finfo(dtype).min, device=device, dtype=dtype)
        mask = torch.triu(mask, diagonal=1)
        return mask.unsqueeze(0).unsqueeze(0).expand(batch_size, 1, seq_len, seq_len)

    def _build_attention_mask(self, attention_mask, dtype):
        batch_size, seq_len = attention_mask.shape
        expanded = attention_mask[:, None, None, :].expand(batch_size, 1, seq_len, seq_len).to(dtype)
        inverted = 1.0 - expanded
        return inverted.masked_fill(inverted.bool(), torch.finfo(dtype).min)

    def get_text_features(self):
        input_ids = self.input_ids
        attention_mask = self.attention_mask

        token_embeds = self.clip_model.text_model.embeddings.token_embedding(input_ids).clone()
        if self.class_specific_context:
            learned_ctx = self.ctx
        else:
            learned_ctx = self.ctx.unsqueeze(0).expand(token_embeds.size(0), -1, -1)
        token_embeds[:, 1:1 + self.n_ctx, :] = learned_ctx

        seq_len = token_embeds.size(1)
        position_ids = torch.arange(seq_len, device=token_embeds.device).unsqueeze(0).expand(token_embeds.size(0), -1)
        hidden_states = token_embeds + self.clip_model.text_model.embeddings.position_embedding(position_ids)

        encoder_outputs = self.clip_model.text_model.encoder(
            inputs_embeds=hidden_states,
            attention_mask=self._build_attention_mask(attention_mask, hidden_states.dtype),
            causal_attention_mask=self._build_causal_mask(token_embeds.size(0), seq_len, hidden_states.dtype, token_embeds.device),
            return_dict=True,
        )
        last_hidden_state = encoder_outputs.last_hidden_state if hasattr(encoder_outputs, "last_hidden_state") else encoder_outputs[0]
        last_hidden_state = self.clip_model.text_model.final_layer_norm(last_hidden_state)

        eos_positions = input_ids.argmax(dim=-1)
        pooled = last_hidden_state[torch.arange(last_hidden_state.size(0), device=last_hidden_state.device), eos_positions]
        text_features = self.clip_model.text_projection(pooled)
        return F.normalize(text_features, dim=-1)

class CoOpClassifier(nn.Module):
    def __init__(self, class_names, clip_model, tokenizer, ctx_init):
        super().__init__()
        self.prompt_learner = CoOpPromptLearner(class_names, clip_model, tokenizer, ctx_init=ctx_init)
        self.logit_scale = nn.Parameter(clip_model.logit_scale.detach().clone())

    def forward(self, input_features):
        sample_features = F.normalize(input_features, dim=-1)
        text_features = self.prompt_learner.get_text_features()
        logit_scale = self.logit_scale.exp().clamp(max=100)
        return logit_scale * sample_features @ text_features.T

def train_coop_classifier(X_train, y_train, X_val, y_val, variant_name, ctx_init, lr, epochs=COOP_EPOCHS):
    coop_model = CoOpClassifier(label_names, clip_model, clip_processor.tokenizer, ctx_init=ctx_init).to(device)
    optimizer = torch.optim.AdamW(
        [p for p in coop_model.parameters() if p.requires_grad],
        lr=lr,
        weight_decay=COOP_WEIGHT_DECAY,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss(label_smoothing=COOP_LABEL_SMOOTHING)

    train_ds = EmbeddingDataset(X_train, y_train)
    train_loader = DataLoader(train_ds, batch_size=min(COOP_BATCH_SIZE, len(train_ds)), shuffle=True)

    X_train_device = X_train.to(device)
    X_val_device = X_val.to(device)
    y_val_device = y_val.to(device)

    best_state = None
    best_val_f1 = -float("inf")

    for ep in range(epochs):
        coop_model.train()
        running_loss = 0.0

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            logits = coop_model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * xb.size(0)

        scheduler.step()

        coop_model.eval()
        with torch.no_grad():
            val_preds = torch.argmax(coop_model(X_val_device), dim=1).cpu().numpy()
            val_acc = accuracy_score(y_val_device.cpu().numpy(), val_preds)
            val_f1m = f1_score(y_val_device.cpu().numpy(), val_preds, average="macro")
            train_preds = torch.argmax(coop_model(X_train_device), dim=1).cpu().numpy()
            train_acc = accuracy_score(y_train.cpu().numpy(), train_preds)

        if val_f1m > best_val_f1:
            best_val_f1 = val_f1m
            best_state = {k: v.detach().cpu().clone() for k, v in coop_model.state_dict().items()}

        if ep % 10 == 0 or ep == epochs - 1:
            mean_loss = running_loss / len(train_ds)
            current_lr = scheduler.get_last_lr()[0]
            print(f"[{variant_name}] Epoch {ep+1} loss={mean_loss:.4f} train_acc={train_acc:.4f} val_acc={val_acc:.4f} val_f1m={val_f1m:.4f} lr={current_lr:.2e}")

    if best_state is not None:
        coop_model.load_state_dict(best_state)

    return coop_model

X_train_mm_coop = make_coop_multimodal_features(X_train_img, X_train_txt)
X_val_mm_coop = make_coop_multimodal_features(X_val_img, X_val_txt)
X_test_mm_coop = make_coop_multimodal_features(X_test_img, X_test_txt)

print("Few-shot CoOp Image")
coop_img_clf = train_coop_classifier(
    X_train_img,
    y_train,
    X_val_img,
    y_val,
    variant_name="Image",
    ctx_init=COOP_CTX_INIT_BY_MODALITY["Image"],
    lr=COOP_LR_BY_MODALITY["Image"],
)
with torch.no_grad():
    coop_img_preds = torch.argmax(coop_img_clf(X_test_img.to(device)), dim=1).cpu().numpy()
coop_img = eval_metrics(y_test.numpy(), coop_img_preds, "Few-shot CoOp Image")

print("Few-shot CoOp Text")
coop_txt_clf = train_coop_classifier(
    X_train_txt,
    y_train,
    X_val_txt,
    y_val,
    variant_name="Text",
    ctx_init=COOP_CTX_INIT_BY_MODALITY["Text"],
    lr=COOP_LR_BY_MODALITY["Text"],
)
with torch.no_grad():
    coop_txt_preds = torch.argmax(coop_txt_clf(X_test_txt.to(device)), dim=1).cpu().numpy()
coop_txt = eval_metrics(y_test.numpy(), coop_txt_preds, "Few-shot CoOp Text")

print("Few-shot CoOp Multimodal")
coop_mm_clf = train_coop_classifier(
    X_train_mm_coop,
    y_train,
    X_val_mm_coop,
    y_val,
    variant_name="Multimodal",
    ctx_init=COOP_CTX_INIT_BY_MODALITY["Multimodal"],
    lr=COOP_LR_BY_MODALITY["Multimodal"],
)
with torch.no_grad():
    coop_mm_preds = torch.argmax(coop_mm_clf(X_test_mm_coop.to(device)), dim=1).cpu().numpy()
coop_mm = eval_metrics(y_test.numpy(), coop_mm_preds, "Few-shot CoOp Multimodal")


Few-shot CoOp Image
[Image] Epoch 1 loss=1.5670 train_acc=0.6862 val_acc=0.6219 val_f1m=0.6124 lr=1.98e-03
[Image] Epoch 11 loss=0.9967 train_acc=0.8692 val_acc=0.6346 val_f1m=0.6259 lr=3.31e-04
[Image] Epoch 15 loss=0.9239 train_acc=0.8931 val_acc=0.6297 val_f1m=0.6221 lr=0.00e+00
Few-shot CoOp Image Acc=0.6515 F1_macro=0.6469 F1_weighted=0.6553
Few-shot CoOp Text
[Text] Epoch 1 loss=1.5014 train_acc=0.7115 val_acc=0.6615 val_f1m=0.6592 lr=9.89e-04
[Text] Epoch 11 loss=0.9448 train_acc=0.8900 val_acc=0.6933 val_f1m=0.6655 lr=1.65e-04
[Text] Epoch 15 loss=0.8897 train_acc=0.9008 val_acc=0.6988 val_f1m=0.6734 lr=0.00e+00
Few-shot CoOp Text Acc=0.7115 F1_macro=0.6967 F1_weighted=0.7138
Few-shot CoOp Multimodal
[Multimodal] Epoch 1 loss=1.4588 train_acc=0.7431 val_acc=0.6637 val_f1m=0.6597 lr=9.89e-04
[Multimodal] Epoch 11 loss=0.9101 train_acc=0.9046 val_acc=0.7150 val_f1m=0.7053 lr=1.65e-04
[Multimodal] Epoch 15 loss=0.8639 train_acc=0.9131 val_acc=0.7150 val_f1m=0.7085 lr=0.00e+00
Few-

# SUMMARY

In [58]:
# =========================
# Cell 15 - Final summary table and save metrics
# =========================
summary = pd.DataFrame([
    ["Zero-shot", "Image", zs_img[0], zs_img[1], zs_img[2]],
    ["Zero-shot", "Text", zs_txt[0], zs_txt[1], zs_txt[2]],
    ["Zero-shot", "Multimodal", zs_mm[0], zs_mm[1], zs_mm[2]],
    ["Few-shot", "Image", fs_img[0], fs_img[1], fs_img[2]],
    ["Few-shot", "Text", fs_txt[0], fs_txt[1], fs_txt[2]],
    ["Few-shot", "Multimodal", fs_mm[0], fs_mm[1], fs_mm[2]],
    ["Few-shot (CoOp)", "Image", coop_img[0], coop_img[1], coop_img[2]],
    ["Few-shot (CoOp)", "Text", coop_txt[0], coop_txt[1], coop_txt[2]],
    ["Few-shot (CoOp)", "Multimodal", coop_mm[0], coop_mm[1], coop_mm[2]],
], columns=["Setting", "Modality", "Accuracy", "F1_macro", "F1_weighted"])

display(summary)
summary.to_csv(os.path.join(OUTPUT_DIR, "clip_results_summary.csv"), index=False)
print("Saved clip_results_summary.csv")

,Setting,Modality,Accuracy,F1_macro,F1_weighted
0,Zero-shot,Image,0.537763,0.511722,0.536804
1,Zero-shot,Text,0.512480,0.500939,0.493999
2,Zero-shot,Multimodal,0.587358,0.569593,0.577619
3,Few-shot,Image,0.675203,0.656185,0.678894
4,Few-shot,Text,0.729335,0.703095,0.728645
5,Few-shot,Multimodal,0.732901,0.718881,0.734847
6,Few-shot (CoOp),Image,0.651540,0.646884,0.655310
7,Few-shot (CoOp),Text,0.711507,0.696707,0.713770
8,Few-shot (CoOp),Multimodal,0.724797,0.721398,0.726513


Saved clip_results_summary.csv
